In [1]:
import pandas as pd
import json
import cv2
import numpy as np
from pathlib import Path
import os

In [ ]:
video_path = r"/home/share/schaer2/idtracking_keypoint/test_8124/8124_34_19-05_23-00.mp4"
bbox = r"/home/share/schaer2/idtracking_keypoint/test_8124/8124_34_19-05_23-00_bbox_0-end_20250715_152103.json"
skpoint = r"/home/share/schaer2/idtracking_keypoint/test_8124/results_skeleton_8124_34_19-05_23-00.json"
timeline_behavior = r"/home/share/schaer2/idtracking_keypoint/output/8137_data_Timelight_segment.csv"

In [3]:
timeline_behavior = pd.read_csv(timeline_behavior)
timeline_behavior

,signal_name,y_pos,start_time,duration,power,frequency,category
0,right_shoulder_angular_vel,0,535.20,12.36,179496.128800,0.554625,Angular Velocities
1,right_shoulder_angular_vel,0,552.92,16.32,377587.946748,0.513617,Angular Velocities
2,right_shoulder_angular_vel,0,651.36,6.36,320875.834919,0.513617,Angular Velocities
3,right_shoulder_angular_vel,0,723.76,8.76,238020.652526,0.323952,Angular Velocities
4,right_shoulder_angular_vel,0,1765.00,5.88,145485.384469,0.949549,Angular Velocities
...,...,...,...,...,...,...,...
94,left_wrist_distance_from_shoulder,27,1241.16,13.80,94543.967022,0.300000,Distance Measures
95,wrist_to_wrist_distance,28,317.64,16.68,283415.463572,0.407906,Distance Measures
96,wrist_to_wrist_distance,28,543.84,12.56,237725.409731,0.377747,Distance Measures
97,wrist_to_wrist_distance,28,562.56,17.24,115367.186086,0.300000,Distance Measures


In [ ]:
timeline_behavior = pd.read_csv(timeline_behavior)
timeline_behavior

,behavior_category,behavior,modifier,event_type,start,end
0,Gestures,Rep-mov,Bras Gauche,state,150.04,151.56
1,Gestures,Rep-mov,Bras Gauche,state,328.80,332.44
2,Gestures,Rep-mov,Bras Gauche,state,553.08,555.24
3,Gestures,Rep-mov,Bras Gauche,state,564.96,567.72
4,Gestures,Rep-mov,Bras Gauche,state,1022.56,1025.32
5,Gestures,Rep-mov,Bras Gauche,state,1086.36,1087.96
6,Gestures,Rep-mov,Bras Gauche,state,1231.80,1233.52
7,Gestures,Rep-mov,Bras Gauche,state,1241.84,1244.56
8,Gestures,Rep-mov,Bras Droit,state,196.00,197.96
9,Gestures,Rep-mov,Bras Droit,state,320.96,330.60


In [ ]:
# Clean implementation for 3 individuals per frame


# OpenPose COCO connections for skeleton
COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

# Colors for 3 individuals (BGR format)
COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]  # Blue, Green, Red

def draw_skeleton(image, keypoints, connections, color, threshold=0.3):
    """Draw skeleton keypoints and connections"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if keypoints is None or len(keypoints) == 0:
        return img
    
    # Draw connections
    for connection in connections:
        idx1, idx2 = connection
        if (idx1 < len(keypoints) and idx2 < len(keypoints) and 
            keypoints[idx1][2] > threshold and keypoints[idx2][2] > threshold):
            x1, y1 = int(keypoints[idx1][0]), int(keypoints[idx1][1])
            x2, y2 = int(keypoints[idx2][0]), int(keypoints[idx2][1])
            cv2.line(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw keypoints
    for x, y, conf in keypoints:
        if conf > threshold:
            cv2.circle(img, (int(x), int(y)), 4, color, -1)
    
    return img

def draw_bbox(image, bbox, color, is_normalized=True):
    """Draw bounding box"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if bbox is None:
        return img

    x1 = bbox[0]
    y1 = bbox[1]
    x2 = bbox[2]
    y2 = bbox[3]
    
    if is_normalized:
        # Convert normalized coordinates to pixel values
        x1 = int(x1 * w)
        y1 = int(y1 * h)
        x2 = int(x2 * w)
        y2 = int(y2 * h)
        
        # Ensure coordinates are within image bounds
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w - 1))
        y2 = max(0, min(y2, h - 1))

    
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    return img

def extract_frame_data(bbox_data, keypoint_data, frame_idx):
    """Extract bbox and keypoints for all individuals in a frame"""
    # Extract bboxes
    bboxes = []
    if str(frame_idx) in bbox_data:
        frame_bboxes = bbox_data[str(frame_idx)]
        for person_id in ['0', '1', '2']:  # 3 individuals
            if person_id in frame_bboxes:
                bboxes.append(frame_bboxes[person_id])
            else:
                bboxes.append(None)
    
    # Extract keypoints
    keypoints_list = []
    if 'instance_info' in keypoint_data:
        # Find frame data
        frame_data = None
        for data in keypoint_data['instance_info']:
            if data['frame_id'] == frame_idx:
                frame_data = data
                break
        
        if frame_data and 'instances' in frame_data:
            instances = frame_data['instances']
            for i in range(3):  # 3 individuals
                if i < len(instances):
                    person = instances[i]
                    if 'keypoints' in person and 'keypoint_scores' in person:
                        kp = np.array(person['keypoints'])  # (17, 2)
                        scores = np.array(person['keypoint_scores'])  # (17,)
                        # Combine to (17, 3) format
                        combined = np.zeros((17, 3))
                        combined[:, :2] = kp
                        combined[:, 2] = scores
                        keypoints_list.append(combined)
                    else:
                        keypoints_list.append(None)
                else:
                    keypoints_list.append(None)
    
    return bboxes, keypoints_list

def create_2x2_video(video_path, bbox_data, keypoint_data, output_path, max_seconds=10):
    """Create 2x2 grid video with all 3 individuals"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    if max_seconds <= 0:
        max_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    else:
        max_frames = int(fps * max_seconds)
    print(f"Processing {max_frames} frames ({max_seconds}s) at {fps} fps")
    
    # Output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width * 2, height * 2))
    
    for frame_idx in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Get data for all 3 individuals
        bboxes, keypoints_list = extract_frame_data(bbox_data, keypoint_data, frame_idx)
        
        # Create 4 quadrants
        top_left = frame.copy()  # Original
        
        top_right = frame.copy()  # With bboxes
        for i, (bbox, color) in enumerate(zip(bboxes, COLORS)):
            if bbox is not None:
                top_right = draw_bbox(top_right, bbox, color)
        
        bottom_left = np.zeros_like(frame)  # Keypoints on black
        for i, (keypoints, color) in enumerate(zip(keypoints_list, COLORS)):
            if keypoints is not None:
                bottom_left = draw_skeleton(bottom_left, keypoints, COCO_CONNECTIONS, color)
        
        bottom_right = frame.copy()  # Keypoints on video
        for i, (keypoints, color) in enumerate(zip(keypoints_list, COLORS)):
            if keypoints is not None:
                bottom_right = draw_skeleton(bottom_right, keypoints, COCO_CONNECTIONS, color)
        
        # Combine into 2x2 grid
        top_row = np.hstack((top_left, top_right))
        bottom_row = np.hstack((bottom_left, bottom_right))
        grid = np.vstack((top_row, bottom_row))
        
        out.write(grid)
        
        if frame_idx % 50 == 0:
            print(f"Processed {frame_idx}/{max_frames} frames")
    
    cap.release()
    out.release()
    print(f"✅ Video saved: {output_path}")

# Load data and create test video
print("Loading data...")
with open(bbox, 'r') as f:
    bbox_data = json.load(f)
with open(skpoint, 'r') as f:
    keypoint_data = json.load(f)

# Create output path
output_path = str(Path(skpoint).parent / "test_3individuals_10s.mp4")

print("Creating 10-second test video with 3 individuals...")
create_2x2_video(video_path, bbox_data, keypoint_data, output_path, max_seconds=10)

Loading data...
Creating 10-second test video with 3 individuals...
Processing 45000 frames (-1s) at 25.0 fps
Processed 0/45000 frames
Processed 50/45000 frames
Creating 10-second test video with 3 individuals...
Processing 45000 frames (-1s) at 25.0 fps
Processed 0/45000 frames
Processed 50/45000 frames
Processed 100/45000 frames
Processed 150/45000 frames
Processed 100/45000 frames
Processed 150/45000 frames
Processed 200/45000 frames
Processed 250/45000 frames
Processed 200/45000 frames
Processed 250/45000 frames
Processed 300/45000 frames
Processed 350/45000 frames
Processed 300/45000 frames
Processed 350/45000 frames
Processed 400/45000 frames
Processed 450/45000 frames
Processed 400/45000 frames
Processed 450/45000 frames
Processed 500/45000 frames
Processed 550/45000 frames
Processed 500/45000 frames
Processed 550/45000 frames
Processed 600/45000 frames
Processed 650/45000 frames
Processed 600/45000 frames
Processed 650/45000 frames
Processed 700/45000 frames
Processed 750/45000

In [14]:
# Verify the output
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"\n✅ Success! Video created with 3 individuals:")
    print(f"📁 File: {output_path}")
    print(f"📏 Size: {file_size / (1024*1024):.2f} MB")
    
    # Check video properties
    cap = cv2.VideoCapture(output_path)
    test_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    test_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    test_fps = cap.get(cv2.CAP_PROP_FPS)
    test_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    print(f"🎥 Properties: {test_width}x{test_height} @ {test_fps} fps")
    print(f"⏱️  Duration: {test_frames/test_fps:.1f} seconds ({test_frames} frames)")
    
    print(f"\n🎨 Color coding:")
    print(f"   🔵 Person 1: Blue")
    print(f"   🟢 Person 2: Green")
    print(f"   🔴 Person 3: Red")
    
    print(f"\n📺 Layout:")
    print(f"   Top-left: Original video")
    print(f"   Top-right: Video + bounding boxes (3 colors)")
    print(f"   Bottom-left: Keypoints only (3 colors)")
    print(f"   Bottom-right: Video + keypoints (3 colors)")
else:
    print("❌ Video creation failed!")


✅ Success! Video created with 3 individuals:
📁 File: /home/share/schaer2/idtracking_keypoint/output/test_3individuals_10s.mp4
📏 Size: 2.19 MB
🎥 Properties: 960x540 @ 25.0 fps
⏱️  Duration: 10.0 seconds (250 frames)

🎨 Color coding:
   🔵 Person 1: Blue
   🟢 Person 2: Green
   🔴 Person 3: Red

📺 Layout:
   Top-left: Original video
   Top-right: Video + bounding boxes (3 colors)
   Bottom-left: Keypoints only (3 colors)
   Bottom-right: Video + keypoints (3 colors)


In [15]:
# Uncomment and run this cell to create the full video (all frames)
# WARNING: This will take much longer and create a large file

# full_output_path = str(Path(skpoint).parent / (Path(skpoint).stem + "_2x2_grid_FULL.mp4"))
# print("Creating FULL video with all 3 individuals...")
# print("This may take several minutes...")
# 
# # Get total video duration first
# cap = cv2.VideoCapture(video_path)
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# fps = cap.get(cv2.CAP_PROP_FPS)
# duration = total_frames / fps
# cap.release()
# 
# print(f"Full video: {duration:.1f} seconds ({total_frames} frames)")
# create_2x2_video(video_path, bbox_data, keypoint_data, full_output_path, max_seconds=int(duration)+1)

print("🎬 Ready! Review the 10-second test video first.")
print("If it looks good, uncomment the code above to create the full video.")

🎬 Ready! Review the 10-second test video first.
If it looks good, uncomment the code above to create the full video.
